# Python Libraries for Machine Learning — SIW Code
Kayes Md Golam Murtuja

The six practical code examples from the SIW report. Run the cells from top to bottom. Install dependencies using `pip install -r requirements.txt`.

## 01_scikit_learn_logistic_regression.py

In [ ]:
"""
Listing 1: Logistic regression pipeline with cross-validation in Scikit-learn.
Dataset: Breast Cancer Wisconsin (included with scikit-learn).
"""

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load the dataset
X, y = load_breast_cancer(return_X_y=True)

# Split the data into training and testing sets
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Build a pipeline: standardisation followed by logistic regression
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

# Evaluate with 5-fold cross-validation on the training data
cv = cross_val_score(model, X_tr, y_tr, cv=5)

# Train on the full training set and evaluate on the test set
model.fit(X_tr, y_tr)

print(f"CV accuracy:   {cv.mean():.4f} +/- {cv.std():.4f}")
print(f"Test accuracy: {model.score(X_te, y_te):.4f}")


## 02_tensorflow_gradient_descent.py

In [ ]:
"""
Listing 2: Gradient descent with tf.GradientTape and tf.function.
Task: Estimate y = wx + b from simulated data.
"""

import numpy as np
import tensorflow as tf

# Generate data: y = 3x + 2 + noise
rng = np.random.default_rng(seed=0)
x = rng.uniform(-1, 1, size=(200, 1)).astype("float32")
y = (
    3.0 * x
    + 2.0
    + rng.normal(0, 0.3, size=(200, 1)).astype("float32")
)

# Trainable parameters
w = tf.Variable(0.0)
b = tf.Variable(0.0)


@tf.function
def train_step():
    # Record operations for automatic differentiation
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean(tf.square(w * x + b - y))

    # Calculate gradients
    dw, db = tape.gradient(loss, [w, b])

    # Update parameters
    w.assign_sub(0.1 * dw)
    b.assign_sub(0.1 * db)

    return loss


# Train for 500 steps
for step in range(500):
    loss = train_step()

print(
    f"w = {w.numpy():.4f}, "
    f"b = {b.numpy():.4f}, "
    f"MSE = {loss.numpy():.4f}"
)


## 03_pytorch_autograd_training.py

In [ ]:
"""
Listing 3: Automatic differentiation and an explicit training loop in PyTorch.
Part (a): Differentiate x^2 + sin(x) at x = 2.
Part (b): Fit a linear regression model to simulated data.
"""

import numpy as np
import torch
from torch import nn

# Part (a): Automatic differentiation
x0 = torch.tensor(2.0, requires_grad=True)
(x0 ** 2 + torch.sin(x0)).backward()
print(f"f'(2) = {x0.grad.item():.4f}")  # 2x + cos(x) at x=2

# Part (b): Linear regression using nn.Linear and SGD
rng = np.random.default_rng(seed=0)
x = rng.uniform(-1, 1, size=(200, 1)).astype("float32")
y = (
    3.0 * x
    + 2.0
    + rng.normal(0, 0.3, size=(200, 1)).astype("float32")
)

# Convert NumPy arrays to PyTorch tensors
x = torch.from_numpy(x)
y = torch.from_numpy(y)

# Define model and optimiser
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Training loop
for epoch in range(500):
    optimizer.zero_grad()                       # Reset gradients
    loss = nn.functional.mse_loss(model(x), y)  # Forward pass
    loss.backward()                              # Backpropagation
    optimizer.step()                             # Update parameters

print(
    f"w = {model.weight.item():.4f}, "
    f"b = {model.bias.item():.4f}"
)


## 04_keras_neural_network_classifier.py

In [ ]:
"""
Listing 4: A Sequential Keras classifier with dropout and early stopping.
Dataset: Breast Cancer Wisconsin (included with scikit-learn).

This is a standalone version of the report's example. It prepares and
standardises the data before training the Keras model.
"""

import keras
from keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Reproducibility
keras.utils.set_random_seed(42)

# Load and split the dataset
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Standardise using training data only
scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)

# Define a Sequential neural network
model = keras.Sequential([
    keras.Input(shape=(30,)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])

# Configure training
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Stop training when validation loss stops improving
stop = keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

# Train and evaluate
model.fit(
    X_tr,
    y_tr,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[stop],
    verbose=0
)

print("Parameters:", model.count_params())
print(
    "Test accuracy: %.4f"
    % model.evaluate(X_te, y_te, verbose=0)[1]
)


## 05_jupyter_model_comparison.py

In [ ]:
"""
Listing 5: Data preparation, training, and evaluation in a notebook-style
workflow. Compares Logistic Regression, SVM, and Random Forest.

The report presents this as three Jupyter Notebook cells. It is provided
here as one Python script so it can also be run from a terminal.
"""

import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Cell 1 and Cell 2: Load and prepare data
X, y = load_breast_cancer(return_X_y=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Fit the scaler only on the training set to prevent data leakage
scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)

# Cell 3: Construct, train, and evaluate models
models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "SVM (RBF kernel)": SVC(random_state=42),
    "Random forest": RandomForestClassifier(300, random_state=42),
}

rows = []

for name, model in models.items():
    model.fit(X_tr, y_tr)

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_te)
    else:
        scores = model.predict_proba(X_te)[:, 1]

    rows.append([
        name,
        accuracy_score(y_te, model.predict(X_te)),
        roc_auc_score(y_te, scores),
    ])

results = pd.DataFrame(
    rows,
    columns=["Model", "Accuracy", "ROC AUC"]
)

print(results.round(4).to_string(index=False))


## 06_ar1_time_series_forecasting.py

In [ ]:
"""
Listing 6: Simulation and forecasting of an AR(1) process.

The process is X_t = 0.8 X_(t-1) + epsilon_t, where epsilon_t is
standard normal noise. The first 800 observations are used for training;
the remaining observations are used for testing in chronological order.
"""

import numpy as np
from sklearn.linear_model import LinearRegression

# Simulate an AR(1) process with phi = 0.8 and sigma = 1
rng = np.random.default_rng(seed=1)
x = np.zeros(1000)

for t in range(1, 1000):
    x[t] = 0.8 * x[t - 1] + rng.normal()

# Use the previous value to predict the next value
X_lag = x[:-1].reshape(-1, 1)
y_next = x[1:]

# Train using the first 800 observations (time-ordered split)
fit = LinearRegression().fit(X_lag[:800], y_next[:800])

# Evaluate on the remaining observations
err = y_next[800:] - fit.predict(X_lag[800:])

print(
    f"phi_hat = {fit.coef_[0]:.4f}, "
    f"forecast RMSE = {np.sqrt(np.mean(err ** 2)):.4f}"
)
